### Rename Columns and Change Data Type

###

In [ ]:
from pyspark.sql.functions import *

df_silver = (
    df.withColumnRenamed("asin", "product_id")
      .withColumnRenamed("sentence", "feedback")
      .withColumnRenamed("helpful", "helpful_score")
      .withColumnRenamed("product_title", "product_name")
      .withColumn("helpful_score", col("helpful_score").cast("double"))
)

df_silver.printSchema()

### Count Null Values

In [ ]:
df_silver.select(
    [count(when(col(c).isNull(), c)).alias(c)
     for c in df_silver.columns]
).show()

### Remove Extra Spaces

In [ ]:
df_silver = df_silver.withColumn(
    "feedback",
    trim(col("feedback"))
)
df_silver.display()

### Remove Empty Feedback

In [ ]:
df_silver = df_silver.filter(
    col("feedback").isNotNull() &
    (trim(col("feedback")) != "")
)

### Create Sentence Length

In [ ]:
df_silver = df_silver.withColumn(
    "sentence_length",
    length(col("feedback"))
)
df.display()

### Create Word Count

In [ ]:
df_silver = df_silver.withColumn(
    "word_count",
    size(split(col("feedback"), " "))
)

### Add Source Information

In [ ]:
df_silver = df_silver.withColumn(
    "ingestion_source",
    lit("Batch")
)

### Add Load Timestamp

In [ ]:
df_silver = df_silver.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

### Check Helpful Score Range

In [ ]:
df_silver.select(
    min("helpful_score"),
    max("helpful_score")
).show()

### Create Helpfulness Bands

In [ ]:
df_silver = df_silver.withColumn(
    "helpfulness_band",
    when(col("helpful_score") < 0.66, "Low")
    .when(col("helpful_score") < 1.33, "Medium")
    .otherwise("High")
)

### Data Quality Check

In [ ]:
df_silver = df_silver.withColumn(
    "score_flag",
    when(
        (col("helpful_score") < 0) |
        (col("helpful_score") > 2),
        "Out Of Range"
    ).otherwise("Valid")
)

### Create Feedback Hash

In [ ]:
from pyspark.sql.window import Window

df_silver = df_silver.withColumn(
    "feedback_hash",
    sha2(col("feedback"), 256)
)

### Create Window Specification

In [ ]:
window_spec = Window.partitionBy("feedback_hash").orderBy("product_id")

### Remove Duplicate Reviews

In [ ]:
df_silver = (
    df_silver
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

### Verify Final Schema

In [ ]:
df_silver.printSchema()